# 01 — Setup + Fine-tuning StyleGAN2 on CelebA-HQ

**Kaggle** — GPU T4 x2, Internet ON.

## Datasets to add:
1. `arnaud58/flickrfaceshq-dataset-ffhq`
2. `lamsimon/celebahq`
3. `atulanandjha/lfwpeople`

## 1. Clean workspace + install PyTorch 2.5 (CUDA ops need it)

In [ ]:
import shutil, glob, os

for path in [
    "/kaggle/working/stylegan2-ada-pytorch",
    "/kaggle/working/training-runs",
    "/kaggle/working/celebahq_256.zip",
    "/kaggle/working/celebahq_512.zip",
    "/kaggle/working/ffhq.pkl",
    "/kaggle/working/ffhq512.pkl",
    "/kaggle/working/stylegan2_celebahq_finetuned.pkl",
] + glob.glob("/kaggle/working/*.png"):
    if os.path.exists(path):
        shutil.rmtree(path) if os.path.isdir(path) else os.remove(path)
        print(f"Removed: {path}")

!df -h /kaggle/working | tail -1
print("Clean.")

In [ ]:
# Downgrade PyTorch so StyleGAN2 CUDA custom ops (upfirdn2d, bias_act) compile.
# Kaggle's PyTorch 2.10+cu128 breaks the JIT build. 2.5.1+cu124 works.
!pip install torch==2.5.1+cu124 torchvision==0.20.1+cu124 --index-url https://download.pytorch.org/whl/cu124 -q

In [ ]:
import torch
assert torch.cuda.is_available()
print(f"PyTorch: {torch.__version__}")
print(f"GPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  [{i}] {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")

In [ ]:
FFHQ_DIR = "/kaggle/input/datasets/arnaud58/flickrfaceshq-dataset-ffhq"
CELEBA_DIR = "/kaggle/input/datasets/lamsimon/celebahq/celeba_hq"
LFW_DIR = "/kaggle/input/datasets/atulanandjha/lfwpeople"

for name, path in [("FFHQ", FFHQ_DIR), ("CelebA-HQ", CELEBA_DIR), ("LFW", LFW_DIR)]:
    if os.path.exists(path):
        print(f"[OK] {name}: {len(os.listdir(path))} items")
    else:
        print(f"[MISSING] {name}: {path}")

## 2. Clone StyleGAN2-ADA + patch misc.py

In [ ]:
!pip install ninja -q

import sys
STYLEGAN_DIR = "/kaggle/working/stylegan2-ada-pytorch"

!git clone https://github.com/NVlabs/stylegan2-ada-pytorch.git {STYLEGAN_DIR}
sys.path.insert(0, STYLEGAN_DIR)

# Patch misc.py — PyTorch removed dataset arg from IterableDataset.__init__
misc_path = f"{STYLEGAN_DIR}/torch_utils/misc.py"
with open(misc_path, "r") as f:
    content = f.read()
content = content.replace("super().__init__(dataset)", "super().__init__()")
with open(misc_path, "w") as f:
    f.write(content)
print("Patched misc.py")

# Patch grid_sample_gradfix — safety net
grid_fix_path = f"{STYLEGAN_DIR}/torch_utils/ops/grid_sample_gradfix.py"
patch = "import torch\n\nenabled = False\n\ndef grid_sample(input, grid, **kwargs):\n    return torch.nn.functional.grid_sample(input, grid, **kwargs)\n"
with open(grid_fix_path, "w") as f:
    f.write(patch)
print("Patched grid_sample_gradfix.py")
print("Done.")

## 3. Download pretrained FFHQ checkpoint

In [ ]:
import subprocess

PRETRAINED_PKL = "/kaggle/working/ffhq.pkl"
PRETRAINED_URL = "https://nvlabs-fi-cdn.nvidia.com/stylegan2-ada-pytorch/pretrained/ffhq.pkl"
FALLBACK_URL = "https://huggingface.co/justinpinkney/stylegan2-ada-pytorch/resolve/main/ffhq.pkl"

if not os.path.exists(PRETRAINED_PKL):
    print("Downloading (~380 MB)...")
    result = subprocess.run(["wget", "-q", "-O", PRETRAINED_PKL, PRETRAINED_URL])
    if result.returncode != 0 or not os.path.exists(PRETRAINED_PKL) or os.path.getsize(PRETRAINED_PKL) < 1e6:
        print("NVIDIA CDN failed, trying HuggingFace...")
        subprocess.run(["wget", "-q", "-O", PRETRAINED_PKL, FALLBACK_URL], check=True)
    print("Done.")
else:
    print("Already exists.")
print(f"Size: {os.path.getsize(PRETRAINED_PKL) / 1e6:.1f} MB")

## 4. Quick test — generate faces from pretrained model

In [ ]:
import pickle, numpy as np
from PIL import Image
import matplotlib.pyplot as plt

with open(PRETRAINED_PKL, "rb") as f:
    G = pickle.load(f)["G_ema"].cuda().eval()
print(f"Generator: {G.img_resolution}x{G.img_resolution}")

def generate_images(G, seeds, truncation_psi=0.7):
    images = []
    for seed in seeds:
        z = torch.from_numpy(np.random.RandomState(seed).randn(1, G.z_dim)).cuda().float()
        label = torch.zeros([1, G.c_dim], device="cuda") if G.c_dim > 0 else None
        with torch.no_grad():
            img = G(z, label, truncation_psi=truncation_psi)
        img = (img.clamp(-1, 1) + 1) / 2
        img = img[0].permute(1, 2, 0).cpu().numpy()
        images.append(img)
    return images

seeds = [42, 123, 256, 512]
images = generate_images(G, seeds)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Pretrained FFHQ — before fine-tuning")
for ax, img, seed in zip(axes, images, seeds):
    ax.imshow(img); ax.set_title(f"seed={seed}"); ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/pretrained_samples.png", dpi=100)
plt.show()

del G
torch.cuda.empty_cache()

## 5. Prepare CelebA-HQ at 256x256

In [ ]:
DATASET_ZIP = "/kaggle/working/celebahq_256.zip"

if not os.path.exists(DATASET_ZIP):
    !python {STYLEGAN_DIR}/dataset_tool.py \
        --source={CELEBA_DIR} \
        --dest={DATASET_ZIP} \
        --width=256 --height=256 \
        --max-images=5000
    print(f"Dataset zip: {os.path.getsize(DATASET_ZIP) / 1e9:.2f} GB")
else:
    print(f"Already exists: {os.path.getsize(DATASET_ZIP) / 1e9:.2f} GB")

## 6. Fine-tune — 2x T4, 256x256, kimg=200

In [ ]:
OUTPUT_DIR = "/kaggle/working/training-runs"

!python {STYLEGAN_DIR}/train.py \
    --outdir={OUTPUT_DIR} \
    --data={DATASET_ZIP} \
    --resume={PRETRAINED_PKL} \
    --cfg=paper256 \
    --gpus=2 \
    --kimg=200 \
    --snap=2 \
    --freezed=13 \
    --aug=noaug \
    --metrics=none \
    --batch=16

## 7. Compare pretrained vs fine-tuned

In [ ]:
import glob

run_dirs = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*")))
assert run_dirs, "No training runs found!"
latest_run = run_dirs[-1]

pkls = sorted(glob.glob(os.path.join(latest_run, "network-snapshot-*.pkl")))
assert pkls, f"No checkpoints! Contents: {os.listdir(latest_run)}"

FINETUNED_PKL = pkls[-1]
print(f"Checkpoint: {FINETUNED_PKL}")

In [ ]:
with open(PRETRAINED_PKL, "rb") as f:
    G_pre = pickle.load(f)["G_ema"].cuda().eval()
with open(FINETUNED_PKL, "rb") as f:
    G_ft = pickle.load(f)["G_ema"].cuda().eval()

print(f"Pretrained: {G_pre.img_resolution}x{G_pre.img_resolution}")
print(f"Fine-tuned: {G_ft.img_resolution}x{G_ft.img_resolution}")

compare_seeds = [42, 123, 256, 512]
imgs_pre = generate_images(G_pre, compare_seeds)
imgs_ft = generate_images(G_ft, compare_seeds)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Pretrained FFHQ (top) vs Fine-tuned CelebA-HQ (bottom)")
for i, seed in enumerate(compare_seeds):
    axes[0, i].imshow(imgs_pre[i]); axes[0, i].set_title(f"Pre seed={seed}"); axes[0, i].axis("off")
    axes[1, i].imshow(imgs_ft[i]); axes[1, i].set_title(f"FT seed={seed}"); axes[1, i].axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/comparison.png", dpi=100)
plt.show()

del G_pre
torch.cuda.empty_cache()

## 8. Export checkpoint

In [ ]:
EXPORT_PATH = "/kaggle/working/stylegan2_celebahq_finetuned.pkl"
shutil.copy(FINETUNED_PKL, EXPORT_PATH)
print(f"Exported: {EXPORT_PATH} ({os.path.getsize(EXPORT_PATH) / 1e6:.1f} MB)")

sample_seeds = list(range(100, 116))
sample_images = generate_images(G_ft, sample_seeds)

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
fig.suptitle("Fine-tuned StyleGAN2 — 16 samples (psi=0.7)")
for ax, img, seed in zip(axes.flat, sample_images, sample_seeds):
    ax.imshow(img); ax.set_title(f"seed={seed}"); ax.axis("off")
plt.tight_layout()
plt.savefig("/kaggle/working/finetuned_grid.png", dpi=100)
plt.show()

print("Week 1 done. Download checkpoint from Output tab.")